<a href="https://colab.research.google.com/github/avocado-planet/02-LangGraph-Agent/blob/main/02_LangGraph_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langchain-core langchain-openai langchain-community langgraph google-search-results

In [3]:
"""
==============================================================================
LangGraph コンポーネント手組み ReAct Agent
==============================================================================

■ 目的:
  create_agent() や create_react_agent() を使わずに、
  LangGraph の基本コンポーネント（State, Node, Edge, 条件付きEdge）を
  自分で組み立てて ReAct エージェントを構築する。

  「create_agent の中身で何が起きているか」を理解するための学習用コード。

■ LangGraph の基本コンポーネント（このコードで使うもの）:
  1. State（ステート）     : ノード間で共有されるデータ構造
  2. Node（ノード）        : 処理の単位（LLM呼び出し、ツール実行）
  3. Edge（エッジ）        : ノード間の固定接続
  4. Conditional Edge      : 条件に応じて次のノードを動的に選ぶ
  5. START / END           : グラフの開始点と終了点

■ 構築するグラフの構造:

  [START]
     │
     ▼
  ┌──────────┐
  │ LLMノード │◄──────────────────┐
  └────┬─────┘                    │
       │                          │
       ▼                          │
  《条件付きエッジ》               │
  tool_callsがある？              │
       │                          │
    Yes │   No                    │
       │    │                     │
       ▼    ▼                     │
  ┌──────┐  [END]                │
  │ツール│                        │
  │ノード│────────────────────────┘
  └──────┘  （結果をステートに追加して
              LLMノードに戻る）

■ 必要パッケージ:
  pip install langchain-core langchain-openai langchain-community \
              langgraph google-search-results

■ 環境変数:
  OPENAI_API_KEY   : OpenAI API キー（必須）
  SERPAPI_API_KEY   : SerpAPI キー（必須）
==============================================================================
"""

import os
import sys
from typing import Annotated
from typing_extensions import TypedDict
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY")

# ---------------------------------------------------------------------------
# 0. 環境変数チェック
# ---------------------------------------------------------------------------
required_env_vars = ["OPENAI_API_KEY", "SERPAPI_API_KEY"]
missing = [v for v in required_env_vars if not os.environ.get(v)]
if missing:
    print(f"[ERROR] 未設定の環境変数: {', '.join(missing)}")
    sys.exit(1)

# ---------------------------------------------------------------------------
# 1. ライブラリのインポート
# ---------------------------------------------------------------------------
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.utilities import SerpAPIWrapper

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

print("環境変数、ライブラリimport完了！")

環境変数、ライブラリimport完了！


In [4]:
# ============================================================================
# コンポーネント① : State（ステート）の定義
# ============================================================================
#
# State は、グラフ全体で共有される「データの入れ物」。
# 各ノードは State を受け取り、更新した State を返す。
#
# Annotated[list, add_messages] の意味:
#   - list 型で、メッセージのリスト
#   - add_messages = 「上書き」ではなく「追記」する
#     （新しいメッセージを既存のリストに append する）
#
# 例: ノードが {"messages": [new_msg]} を返すと、
#     state["messages"] は [既存msg1, 既存msg2, new_msg] になる
#
# ★ create_agent / create_react_agent が内部で自動生成するステートと
#   同じ構造を、自分で定義している
# ---------------------------------------------------------------------------
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# ============================================================================
# コンポーネント② : ツール（Tool）の定義
# ============================================================================
#
# @tool デコレータを使うと:
#   - 関数の docstring が自動的に description になる
#   - 引数の型アノテーションから JSON Schema が自動生成される
#   - LLM の function calling で使える形式に変換される
# ---------------------------------------------------------------------------
search_api = SerpAPIWrapper()

@tool
def web_search(query: str) -> str:
    """Search the web for current events, recent news, or real-time information.
    Use this when you need up-to-date information that you don't already know."""
    return search_api.run(query)

# ツールのリスト（複数のツールを追加可能）
tools = [web_search]


# ============================================================================
# コンポーネント③ : LLM の設定とツールのバインド
# ============================================================================
#
# bind_tools() で LLM にツールの存在を教える。
# これにより LLM は応答時に tool_call を含めることができるようになる。
#
# ★ これが「新式（tool calling 方式）」の核心。
#   旧式ではプロンプトのテキストにツール情報を埋め込んでいたが、
#   新式では API レベルでツール定義を渡す。
# ---------------------------------------------------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

In [5]:
# ============================================================================
# コンポーネント④ : ノード（Node）の定義
# ============================================================================
#
# ノードは「State を受け取り、更新した State を返す関数」。
# グラフの中で実際の処理を行う単位。
#
# ここでは2つのノードを定義:
#   - llm_node  : LLM を呼び出して次の行動を決める
#   - tool_node : LLM が指定したツールを実行する
# ---------------------------------------------------------------------------

# --- ノード A: LLM ノード ---
# LLM に会話履歴（messages）を渡し、応答を得る。
# 応答には以下の2パターンがある:
#   パターン1: tool_calls あり → ツールの呼び出しを要求
#   パターン2: tool_calls なし → 最終回答（テキスト）
def llm_node(state: AgentState):
    """LLM を呼び出し、次の行動を決定するノード"""

    # システムプロンプトを先頭に追加
    system_msg = SystemMessage(
        content="You are a helpful assistant. Think step by step and use tools when needed."
    )
    # LLM を呼び出し
    response = llm_with_tools.invoke([system_msg] + state["messages"])

    # State の messages に LLM の応答を追記して返す
    return {"messages": [response]}


# --- ノード B: ツールノード ---
# LangGraph が提供する ToolNode を使用。
# 最後の AI メッセージに含まれる tool_calls を自動的に実行し、
# 結果を ToolMessage として返す。
tool_node = ToolNode(tools=tools)

In [6]:
# ============================================================================
# コンポーネント⑤ : グラフの構築（ノード + エッジ）
# ============================================================================
#
# StateGraph にノードを追加し、エッジで接続する。
#
# エッジの種類:
#   1. 通常エッジ (add_edge)      : A → B へ常に遷移
#   2. 条件付きエッジ (add_conditional_edges) : 条件に応じて分岐
#   3. START エッジ : グラフの開始点
#   4. END   : グラフの終了点
# ---------------------------------------------------------------------------

# --- グラフの初期化 ---
graph_builder = StateGraph(AgentState)

# --- ノードの追加 ---
graph_builder.add_node("llm", llm_node)       # LLM ノード
graph_builder.add_node("tools", tool_node)     # ツールノード

# --- エッジの追加 ---

# (1) START → llm : グラフは常に LLM ノードから開始
graph_builder.add_edge(START, "llm")

# (2) llm → ??? : 条件付きエッジ（ここが ReAct の核心！）
#     tools_condition は LangGraph が提供するヘルパー関数で、
#     最後の AI メッセージに tool_calls が含まれているかをチェック:
#       - tool_calls あり → "tools" ノードへ
#       - tool_calls なし → END へ（最終回答）
graph_builder.add_conditional_edges(
    "llm",              # 分岐元のノード
    tools_condition,    # 分岐条件を判定する関数
    {                   # 判定結果 → 遷移先のマッピング
        "tools": "tools",   # tool_calls あり → ツールノードへ　★tools_conditionの戻り値: 遷移先のノード名
        "__end__": END,      # tool_calls なし → 終了　　★tools_conditionの戻り値: 遷移先のノード名
    },
)

# (3) tools → llm : ツール実行後は必ず LLM に戻る（ループ）
#     これにより ReAct の「Action → Observation → Thought」サイクルが実現
graph_builder.add_edge("tools", "llm")


# ============================================================================
# コンポーネント⑥ : グラフのコンパイル
# ============================================================================
#
# compile() でグラフを実行可能な形に変換する。
# ここでバリデーション（未接続ノードのチェック等）も行われる。
# ---------------------------------------------------------------------------
agent = graph_builder.compile()

In [10]:
# ============================================================================
# 実行
# ============================================================================
question = "狛江はどんなところですか？"
print(f"=== 質問 ===\n{question}\n")

try:
    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]}
    )

    # --- 全ステップの表示 ---
    print("=== Agent Response（全ステップ） ===")
    for message in result["messages"]:
        role = message.type

        if hasattr(message, "tool_calls") and message.tool_calls:
            print(f"[{role}] ツール呼び出し:")
            for tc in message.tool_calls:
                print(f"  → {tc['name']}({tc['args']})")
        elif message.content:
            print(f"[{role}] {message.content}")
        print("---")

    # --- 最終回答の抽出 ---
    for message in reversed(result["messages"]):
        if message.type == "ai" and message.content and not getattr(message, "tool_calls", []):
            print(f"\n=== 最終回答 ===\n{message.content}")
            break

except Exception as e:
    print(f"\n[ERROR] {e}")
    raise

=== 質問 ===
狛江はどんなところですか？

=== Agent Response（全ステップ） ===
[human] 狛江はどんなところですか？
---
[ai] 狛江（こまえ）は、日本の東京都に位置する市で、特に多摩地域に属しています。狛江は、自然環境が豊かで、住宅地としても人気があります。以下に狛江の特徴をいくつか挙げます。

1. **交通の便**: 狛江は小田急小田原線が通っており、新宿や渋谷などの都心へのアクセスが良好です。

2. **自然環境**: 多摩川が近くにあり、河川敷や公園が整備されているため、散策やスポーツを楽しむことができます。

3. **歴史と文化**: 狛江には古い神社や寺院があり、地域の歴史を感じることができます。特に狛江神社は有名です。

4. **住環境**: 住宅地としては静かで落ち着いた雰囲気があり、ファミリー層にも人気があります。

5. **商業施設**: 小規模な商業施設や飲食店もあり、日常生活に必要なものは揃っています。

狛江は、都市の便利さと自然の豊かさを兼ね備えた魅力的な場所です。
---

=== 最終回答 ===
狛江（こまえ）は、日本の東京都に位置する市で、特に多摩地域に属しています。狛江は、自然環境が豊かで、住宅地としても人気があります。以下に狛江の特徴をいくつか挙げます。

1. **交通の便**: 狛江は小田急小田原線が通っており、新宿や渋谷などの都心へのアクセスが良好です。

2. **自然環境**: 多摩川が近くにあり、河川敷や公園が整備されているため、散策やスポーツを楽しむことができます。

3. **歴史と文化**: 狛江には古い神社や寺院があり、地域の歴史を感じることができます。特に狛江神社は有名です。

4. **住環境**: 住宅地としては静かで落ち着いた雰囲気があり、ファミリー層にも人気があります。

5. **商業施設**: 小規模な商業施設や飲食店もあり、日常生活に必要なものは揃っています。

狛江は、都市の便利さと自然の豊かさを兼ね備えた魅力的な場所です。
